# DAVE Documents API — Document Routes

**Prerequisite:** run `00_auth_setup.ipynb` first to generate `auth_state.py`.

| Method | Path | Description |
|--------|------|-------------|
| GET | `/api/document` | List / paginate documents |
| GET | `/api/document/{id}` | Get single document |
| GET | `/api/document/{id}/true` | Get document (de-anonymized) |
| POST | `/api/document/by-ids` | Fetch multiple documents by IDs |
| GET | `/api/document/services` | List annotation services |
| POST | `/api/document/services` | Create annotation service |
| PUT | `/api/document/services/{id}` | Update annotation service |
| DELETE | `/api/document/services/{id}` | Delete annotation service |
| GET | `/api/document/configurations` | List pipeline configurations |
| GET | `/api/document/configurations/active` | Get active configuration |
| POST | `/api/document/configurations` | Create configuration |
| PUT | `/api/document/configurations/{id}` | Update configuration |
| POST | `/api/document/configurations/{id}/activate` | Set active configuration |
| DELETE | `/api/document/configurations/{id}` | Delete configuration |

In [ ]:
import sys, os, json, requests

# Add notebooks directory to path so auth_state can be imported
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from auth_state import API_BASE, auth_headers

print(f"API base: {API_BASE}")

## GET /api/document — List documents

In [ ]:
resp = requests.get(
    f"{API_BASE}/document",
    params={"q": "", "page": 1, "limit": 5},
    headers=auth_headers(),
)
resp.raise_for_status()
page = resp.json()
print(f"Total docs: {page.get('totalDocs', 'N/A')}")
docs = page.get("docs", [])
for d in docs[:3]:
    print(f"  id={d.get('id')}  name={d.get('name')}")

# Keep the first ID for subsequent cells
sample_doc_id = docs[0]["id"] if docs else None
print(f"\nUsing sample_doc_id = {sample_doc_id}")

## GET /api/document/{id} — Get single document

In [ ]:
if sample_doc_id is None:
    print("No documents found — skipping.")
else:
    resp = requests.get(
        f"{API_BASE}/document/{sample_doc_id}",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    doc = resp.json()
    print(f"name : {doc.get('name')}")
    print(f"id   : {doc.get('id')}")
    print(f"text snippet: {str(doc.get('text', ''))[:200]}")

## GET /api/document/{id}/true — Get document (de-anonymized)

In [ ]:
if sample_doc_id is None:
    print("No documents found — skipping.")
else:
    resp = requests.get(
        f"{API_BASE}/document/{sample_doc_id}/true",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    doc = resp.json()
    print(f"name : {doc.get('name')}")
    print(f"anonymized feature: {doc.get('features', {}).get('anonymized')}")

## POST /api/document/by-ids — Fetch multiple documents

In [ ]:
ids_to_fetch = [d["id"] for d in docs[:2]] if docs else []

if not ids_to_fetch:
    print("No document IDs available — skipping.")
else:
    resp = requests.post(
        f"{API_BASE}/document/by-ids",
        json={"ids": ids_to_fetch, "deAnonimize": False},
        headers=auth_headers(),
    )
    resp.raise_for_status()
    result = resp.json()
    print(f"Returned {len(result)} documents")
    for d in result:
        print(f"  id={d.get('id')}  name={d.get('name')}")

---
## Annotation Services
### GET /api/document/services — List services

In [ ]:
resp = requests.get(f"{API_BASE}/document/services", headers=auth_headers())
resp.raise_for_status()
services = resp.json()
print(f"{len(services)} service(s) found")
print(json.dumps(services, indent=2))

sample_service_id = services[0]["_id"] if services else None

### POST /api/document/services — Create a new service

In [ ]:
new_service_payload = {
    "name":        "test-ner-service",
    "uri":         "http://localhost:8081",
    "serviceType": "ner",
    "description": "NER annotation service for testing",
}

resp = requests.post(
    f"{API_BASE}/document/services",
    json=new_service_payload,
    headers=auth_headers(),
)
if resp.status_code == 409:
    print("Service already exists — skipping creation")
    created_service = None
else:
    resp.raise_for_status()
    created_service = resp.json()
    print("Created:", json.dumps(created_service, indent=2))

### PUT /api/document/services/{id} — Update a service

In [ ]:
svc_id = (created_service or {}).get("_id") or sample_service_id

if not svc_id:
    print("No service ID available — skipping.")
else:
    resp = requests.put(
        f"{API_BASE}/document/services/{svc_id}",
        json={"description": "Updated description", "disabled": False},
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print("Updated:", json.dumps(resp.json(), indent=2))

### DELETE /api/document/services/{id} — Delete a service

In [ ]:
# Only delete the service we just created; guard with confirmation flag
delete_service = False  # ← set True to actually delete

del_id = (created_service or {}).get("_id")
if delete_service and del_id:
    resp = requests.delete(
        f"{API_BASE}/document/services/{del_id}",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print(resp.json())
else:
    print("Skipped (set delete_service=True to run).")

---
## Pipeline Configurations
### GET /api/document/configurations — List configurations

In [ ]:
resp = requests.get(f"{API_BASE}/document/configurations", headers=auth_headers())
resp.raise_for_status()
configs = resp.json()
print(f"{len(configs)} configuration(s) found")
print(json.dumps(configs, indent=2))

sample_config_id = configs[0]["_id"] if configs else None

### GET /api/document/configurations/active — Active configuration

In [ ]:
resp = requests.get(f"{API_BASE}/document/configurations/active", headers=auth_headers())
if resp.status_code == 404:
    print("No active configuration found.")
else:
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2))

### POST /api/document/configurations — Create configuration

In [ ]:
new_config_payload = {
    "name": "test-config",
    "steps": [
        {"slot": "ner",    "serviceId": None},
        {"slot": "linker", "serviceId": None},
    ],
    "isActive": False,
}

resp = requests.post(
    f"{API_BASE}/document/configurations",
    json=new_config_payload,
    headers=auth_headers(),
)
if resp.status_code == 409:
    print("Configuration already exists — skipping.")
    created_config = None
else:
    resp.raise_for_status()
    created_config = resp.json()
    print("Created:", json.dumps(created_config, indent=2))

### PUT /api/document/configurations/{id} — Update configuration

In [ ]:
cfg_id = (created_config or {}).get("_id") or sample_config_id

if not cfg_id:
    print("No configuration ID — skipping.")
else:
    resp = requests.put(
        f"{API_BASE}/document/configurations/{cfg_id}",
        json={"name": "test-config-updated"},
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2))

### POST /api/document/configurations/{id}/activate — Activate configuration

In [ ]:
cfg_id = (created_config or {}).get("_id") or sample_config_id

if not cfg_id:
    print("No configuration ID — skipping.")
else:
    resp = requests.post(
        f"{API_BASE}/document/configurations/{cfg_id}/activate",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2))

### DELETE /api/document/configurations/{id} — Delete configuration

In [ ]:
delete_config = False  # ← set True to actually delete
del_cfg_id = (created_config or {}).get("_id")

if delete_config and del_cfg_id:
    resp = requests.delete(
        f"{API_BASE}/document/configurations/{del_cfg_id}",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print(resp.json())
else:
    print("Skipped (set delete_config=True to run).")